# Compound Effects Demo

This notebook demonstrates German compound decomposition and how preprocessing can affect translation-oriented metrics.

In [2]:
from pathlib import Path
import os
import sys

import pandas as pd

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'config.py').exists() and (candidate / 'preprocessing').exists():
            return candidate
    raise RuntimeError('Could not locate Machine_Translation project root.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# Ensure absolute imports like `projects...` resolve from the workspace root.
WORKSPACE_ROOT = PROJECT_ROOT.parent.parent
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

from preprocessing.morphological_preprocessing import GermanCompoundPreprocessor
from evaluation.metrics import BLEUMetric

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\Nikolai\OneDrive\Desktop\Portfolio\CS_Language_Portfolio\projects\Machine_Translation


In [3]:
class MockDecomposer:
    def decompose(self, word: str):
        lexicon = {
            'schmetterlingshaus': ['schmetterling', 'haus'],
            'abendessen': ['abend', 'essen'],
            'tageslicht': ['tages', 'licht'],
            'blauwal': ['blau', 'wal']
        }
        return lexicon.get(word.lower().strip('.,!?;:'), [word])

preprocessor = GermanCompoundPreprocessor(decomposer=MockDecomposer(), enable=True)

sentences = [
    'Das Schmetterlingshaus ist sehr gross.',
    'Das Abendessen war lecker.',
    'Ein Blauwal schwimmt im tiefen Wasser.'
]

preprocessed_rows = []
for text in sentences:
    out = preprocessor.preprocess(text)
    preprocessed_rows.append({
        'original': out.original,
        'preprocessed': out.preprocessed,
        'decomposition_count': out.metadata.get('decomposition_count', 0),
        'decompositions': out.decompositions
    })

pd.DataFrame(preprocessed_rows)

,original,preprocessed,decomposition_count,decompositions
0,Das Schmetterlingshaus ist sehr gross.,Das schmetterling|haus ist sehr gross.,1,"{'Das': ['Das'], 'Schmetterlingshaus': ['schme..."
1,Das Abendessen war lecker.,Das abend|essen war lecker.,1,"{'Das': ['Das'], 'Abendessen': ['abend', 'esse..."
2,Ein Blauwal schwimmt im tiefen Wasser.,Ein blau|wal schwimmt im tiefen Wasser.,1,"{'Ein': ['Ein'], 'Blauwal': ['blau', 'wal'], '..."


In [4]:
# Toy metric experiment: compare two hypothetical translations for each source.
metric = BLEUMetric(max_n=4, smooth=True)

toy_eval = [
    {
        'source': 'Das Schmetterlingshaus ist sehr gross.',
        'raw_hyp': 'The butterflyhome is very big.',
        'pre_hyp': 'The butterfly house is very big.',
        'ref': 'The butterfly house is very large.'
    },
    {
        'source': 'Das Abendessen war lecker.',
        'raw_hyp': 'The eveningmeal was tasty.',
        'pre_hyp': 'The evening meal was tasty.',
        'ref': 'Dinner was delicious.'
    },
    {
        'source': 'Ein Blauwal schwimmt im tiefen Wasser.',
        'raw_hyp': 'A bluewhale swims in deep water.',
        'pre_hyp': 'A blue whale swims in deep water.',
        'ref': 'A blue whale swims in deep water.'
    }
]

rows = []
for row in toy_eval:
    bleu_raw = metric.score(row['raw_hyp'], [row['ref']]).score
    bleu_pre = metric.score(row['pre_hyp'], [row['ref']]).score
    rows.append({
        'source': row['source'],
        'bleu_raw': round(bleu_raw, 4),
        'bleu_preprocessed': round(bleu_pre, 4),
        'delta': round(bleu_pre - bleu_raw, 4)
    })

pd.DataFrame(rows)

,source,bleu_raw,bleu_preprocessed,delta
0,Das Schmetterlingshaus ist sehr gross.,0.3161,0.8091,0.4930
1,Das Abendessen war lecker.,0.3593,0.2730,-0.0863
2,Ein Blauwal schwimmt im tiefen Wasser.,0.5447,1.0000,0.4553


In [ ]:
# Optional heavy experiment with real translation models.
RUN_FULL_EXPERIMENT = False

if RUN_FULL_EXPERIMENT:
    from experiments.decomposition_experiment import MorphologicalPreprocessingExperiment

    experiment = MorphologicalPreprocessingExperiment(decomposer=MockDecomposer())
    sources = [row['source'] for row in toy_eval]
    references = [row['ref'] for row in toy_eval]
    comparisons = experiment.compare_batch(sources, references, source_lang='de', target_lang='en')

    report_rows = []
    for comp in comparisons:
        report_rows.append({
            'source': comp.source_text,
            'bleu_raw': comp.bleu_raw,
            'bleu_preprocessed': comp.bleu_preprocessed,
            'improvement_pct': comp.improvement
        })

    pd.DataFrame(report_rows)
else:
    print('Skipped full model experiment. Toggle RUN_FULL_EXPERIMENT to True to run it.')

## Interpretation

- Compound-aware preprocessing can improve lexical segmentation.
- Better segmentation can increase n-gram overlap against references.
- Validate gains with full model experiments and linguistic error analysis.